In [0]:
%run "../00_config"

In [0]:
import json
from datetime import datetime, timezone
from pyspark.sql import functions as F

##Fetch orders

In [0]:
all_orders = [] 

with open(f"{ORDERS_VOLUME}/orders.json", "r") as f:
    orders = json.load(f)

print(f"  Fetched {len(orders)} orders")

for ord in orders:
    all_orders.append({
        "order_id":             ord.get("order_id", ""),
        "customer_id":          ord.get("customer_id", ""),
        "customer_name":        ord.get("customer_name", ""),
        "city":                 ord.get("city", ""),
        "order_date":           ord.get("order_date", ""),
        "quantity":             ord.get("quantity", ""),
        "payment_method":       ord.get("payment_method", ""),
        "order_status":         ord.get("order_status", ""),
        "device_type":          ord.get("device_type", ""),
        "_snapshot_date":       SNAPSHOT_DATE,
        "_ingested_at":         datetime.now(timezone.utc).strftime("%d-%m-%Y %H:%M:%S")
    })

##Write to Bronze

In [0]:
df_bronze_orders = spark.createDataFrame(all_orders)

(df_bronze_orders
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_ORDERS)
)

print(f"✅ {df_bronze_orders.count()} orders written to {BRONZE_ORDERS}")

##Validate

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.bronze.bronze_orders